# ENVI-met Export Demo

This notebook demonstrates how to export a VoxCity model to **ENVI-met INX** format for microclimate simulation.

Contents:
1. Generate or load a VoxCity model
2. Basic INX export (default settings)
3. Generate the plant database (EDB file)
4. Customize land cover → ENVI-met ID mapping
5. Customize building materials (wall & roof) and rooftop vegetation
6. Configure grid settings (telescoping, vertical stretch)
7. Putting it all together
8. Reference: Available soil profile IDs

## 1. Generate or load a VoxCity model

Choose **Option A** to generate a new model, or **Option B** to load a previously saved one.

In [ ]:
# Option A: Generate a new model
from voxcity.generator import get_voxcity

rectangle_vertices = [
    (139.7600, 35.6800),  # Southwest corner (longitude, latitude)
    (139.7600, 35.6900),  # Northwest corner
    (139.7700, 35.6900),  # Northeast corner
    (139.7700, 35.6800),  # Southeast corner
]

meshsize = 5  # meters
voxcity = get_voxcity(rectangle_vertices, meshsize)

In [ ]:
# Option B: Load a previously saved model
# from voxcity.generator.io import load_voxcity
# voxcity = load_voxcity("output/voxcity.h5")

## 2. Basic INX export (default settings)

The simplest usage — export with all default ENVI-met IDs and grid settings.

In [ ]:
from voxcity.exporter.envimet import export_inx, generate_edb_file

output_directory = './output/envimet'
file_basename = 'voxcity'

# Export the INX model file
export_inx(voxcity, output_directory, file_basename)

print(f"INX file saved to {output_directory}/{file_basename}.INX")

## 3. Generate the plant database (EDB file)

ENVI-met needs a plant database (`.edb`) that defines the 3D tree models referenced in the INX file.
`generate_edb_file()` creates `projectdatabase.edb` with trees from 1–50 m height.

Parameters:
- `lad` — Leaf Area Density in m²/m³ (default: 1.0)
- `trunk_height_ratio` — ratio of trunk height to total tree height (default: 11.76/19.98 ≈ 0.59)
- `output_dir` — directory to write the file (default: current directory)

In [ ]:
# Generate with default LAD and save to the same output directory as the INX
generate_edb_file(
    lad=1.0,
    trunk_height_ratio=11.76 / 19.98,
    output_dir='./output/envimet'
)

print("EDB file saved to ./output/envimet/projectdatabase.edb")

In [ ]:
# Example: denser canopy with higher LAD
generate_edb_file(
    lad=2.0,
    output_dir='./output/envimet'
)

### How to use the EDB file in ENVI-met

After generating the `projectdatabase.edb` file, you need to register it as the **project database** in ENVI-met so that the 3D plant definitions (trees) are recognized.

**Steps:**

1. **Place the file** — Copy `projectdatabase.edb` into your ENVI-met project folder (the same folder that contains the `.INX` file), or note its location.

2. **Open ENVI-met** — Launch ENVI-met and open (or create) the project that will use your INX model.

3. **Set the project database** — Go to **Edit → Project Database** (or open the **Database Manager**). Use **File → Open** to load the `projectdatabase.edb` file. This tells ENVI-met where to find the custom tree definitions (e.g., `H05W01`, `H10W01`, …, `H50W01`).

4. **Load the INX model** — Open the generated `.INX` file in ENVI-met SPACES. The 3D plant IDs in the model will now resolve to the tree definitions in your EDB file.

5. **Verify** — In SPACES, check that trees appear correctly by inspecting the 3D plant layer. Each plant ID corresponds to a tree of a specific height (e.g., `H10W01` = 10 m tall tree).

> **Tip:** The EDB file and the INX file must be in the **same project folder** for ENVI-met to automatically discover the plant database. If they are in different locations, you will need to manually load the EDB via the Database Manager.

## 4. Customize land cover → ENVI-met ID mapping

VoxCity uses a standard 14-class land cover classification. When exporting to ENVI-met,
each class is mapped to:
- A **vegetation (simple plant) ID** — the `<ID_plants1D>` grid
- A **soil profile / surface ID** — the `<ID_soilprofile>` grid

You can override any of these defaults using the `envimet_mapping` parameter.
Each key is the **1-based land cover class index**, and the value is a dict with
optional `'veg'` and/or `'mat'` keys.

> **Note:** Tree vegetation (class 5, `'veg'` key) cannot be customized here — trees are 
> handled as 3D plants via the EDB database.

### Default mapping reference

| Class | Land Cover     | Default veg ID | Default mat ID | Description              |
|------:|:---------------|:---------------|:---------------|:-------------------------|
| 1     | Bareland       | _(empty)_      | `000000`       | Default soil             |
| 2     | Rangeland      | `0200XX`       | `000000`       | Grass / default soil     |
| 3     | Shrub          | `0200H1`       | `000000`       | Hedge / default soil     |
| 4     | Agriculture    | `0200XX`       | `000000`       | Grass / default soil     |
| 5     | Tree           | _(3D plant)_   | `000000`       | Handled via EDB          |
| 6     | Moss/lichen    | `0200XX`       | `000000`       | Grass / default soil     |
| 7     | Wetland        | `0200XX`       | `0200WW`       | Grass / water            |
| 8     | Mangrove       | _(empty)_      | `0200WW`       | Water                    |
| 9     | Water          | —              | `0200WW`       | Water                    |
| 10    | Snow/ice       | —              | `000000`       | Default soil             |
| 11    | Developed      | —              | `0200PG`       | Concrete pavement gray   |
| 12    | Road           | —              | `0200ST`       | Asphalt road             |
| 13    | Building       | —              | `000000`       | Default soil             |
| 14    | No Data        | —              | `000000`       | Default soil             |

In [ ]:
# Example: customize specific land cover classes
envimet_mapping = {
    # Class 2 (Rangeland): use a different grass species
    2: {'veg': '0200XX'},

    # Class 11 (Developed): use lighter concrete pavement
    11: {'mat': '0200PL'},   # Concrete Pavement Light

    # Class 12 (Road): use red-coated asphalt
    12: {'mat': '0200AR'},   # Asphalt road with red coating

    # Class 7 (Wetland): change both veg and surface
    7: {'veg': '0200XX', 'mat': '0200WW'},
}

export_inx(
    voxcity,
    output_directory='./output/envimet_custom',
    file_basename='voxcity_custom',
    envimet_mapping=envimet_mapping,
)

print("Custom INX exported.")

## 5. Customize building materials (wall & roof)

ENVI-met applies a **common wall material** and **common roof material** to all buildings.
By default these are `000000` (ENVI-met system default: moderate insulation).

### Common wall material IDs

| ID       | Description                          |
|:---------|:-------------------------------------|
| `000000` | Default Wall - moderate insulation   |
| `0200MI` | Default Wall - moderate insulation   |
| `0200C1` | Concrete Wall (heavy)                |
| `0200C2` | Concrete wall (light weight)         |
| `0200B2` | Brick wall (burned)                  |
| `0200G4` | Clear Float glass (one layered)      |
| `0200GH` | Heat protection glass (one air layer)|

### Common roof material IDs

| ID       | Description                          |
|:---------|:-------------------------------------|
| `000000` | Default Wall - moderate insulation   |
| `0200R1` | Roofing: Tile                        |
| `0200C1` | Concrete Wall (heavy)                |
| `0200AL` | Aluminium (single layer)             |

In [ ]:
# Example: set brick walls with tile roof
export_inx(
    voxcity,
    output_directory='./output/envimet_materials',
    file_basename='voxcity_brick',
    common_wall_material='0200B2',   # Brick wall (burned)
    common_roof_material='0200R1',   # Roofing: Tile
)

print("INX with custom wall/roof materials exported.")

### Rooftop vegetation (green roofs)

By default, 1D vegetation (grass, shrubs, etc.) is **removed** from cells that contain buildings.
This prevents unrealistic ground-level plants from appearing inside building footprints.

Set `rooftop_vegetation=True` to **keep** 1D vegetation on building cells — useful for
modeling green roofs or rooftop gardens in ENVI-met.

In [ ]:
# Example: export with green roofs (keep 1D vegetation on building cells)
export_inx(
    voxcity,
    output_directory='./output/envimet_greenroof',
    file_basename='voxcity_greenroof',
    rooftop_vegetation=True,
)

print("INX with rooftop vegetation exported.")

## 6. Configure grid settings

ENVI-met supports a **telescoping grid** where vertical cell size increases above a certain height,
reducing total cell count while keeping fine resolution near the ground.

Parameters:
- `useTelescoping_grid` (bool) — enable/disable telescoping (default: `False`)
- `verticalStretch` (float) — percentage increase per cell above start height (default: 20%)
- `startStretch` (float) — height (m) to begin stretching (default: auto from max building height)
- `domain_building_max_height_ratio` (float) — ratio of domain height to max building (default: 3)
- `min_grids_Z` (int) — minimum number of vertical grid cells (default: 20)
- `author_name` (str) — author name in the INX metadata
- `model_description` (str) — description in the INX metadata

In [ ]:
# Example: telescoping grid with metadata
export_inx(
    voxcity,
    output_directory='./output/envimet_telescoping',
    file_basename='voxcity_tele',
    useTelescoping_grid=True,
    verticalStretch=20,         # 20% growth per cell
    startStretch=50,            # start stretching at 50m
    domain_building_max_height_ratio=3,
    min_grids_Z=30,
    author_name='VoxCity User',
    model_description='Telescoping grid demo',
)

print("Telescoping grid INX exported.")

## 7. Putting it all together

Combine all customization options — including rooftop vegetation — in a single export call.

In [ ]:
output_dir = './output/envimet_full'

# 1) Export INX with all customizations
export_inx(
    voxcity,
    output_directory=output_dir,
    file_basename='voxcity_full',
    # Land cover → ENVI-met ID overrides
    envimet_mapping={
        11: {'mat': '0200PL'},  # Developed → Concrete Pavement Light
        12: {'mat': '0200AR'},  # Road → Asphalt with red coating
    },
    # Building materials
    common_wall_material='0200B2',
    common_roof_material='0200R1',
    # Rooftop vegetation
    rooftop_vegetation=True,
    # Grid settings
    useTelescoping_grid=True,
    verticalStretch=20,
    min_grids_Z=25,
    # Metadata
    author_name='VoxCity User',
    model_description='Full customization demo',
)

# 2) Generate plant database in the same directory
generate_edb_file(
    lad=1.0,
    output_dir=output_dir,
)

print(f"All files saved to {output_dir}/")

print("  - voxcity_full.INX")
print("  - projectdatabase.edb")

## 8. Reference: Available soil profile IDs

These are the most commonly used ENVI-met soil profile IDs (for `'mat'` in `envimet_mapping`):

| ID       | Description                       |
|:---------|:----------------------------------|
| `000000` | Default Sandy Loam                |
| `0200ST` | Asphalt Road                      |
| `0200PP` | Pavement (Concrete), used/dirty   |
| `0200PG` | Concrete Pavement Gray            |
| `0200PL` | Concrete Pavement Light           |
| `0200PD` | Concrete Pavement Dark            |
| `0200WW` | Deep Water                        |
| `0200SL` | Default Unsealed Soil (Sandy Loam)|
| `0200LO` | Loamy Soil                        |
| `0200SD` | Sandy Soil                        |
| `0200KK` | Brick road (red stones)           |
| `0200KG` | Brick road (yellow stones)        |
| `0200GG` | Dark Granite Pavement             |
| `0200GS` | Granite Pavement (single stones)  |
| `0200G2` | Granite shining                   |
| `0200AR` | Asphalt road with red coating     |
| `0200BA` | Basalt Brick Road                 |
| `0200WD` | Wood Planks                       |
| `0200DW` | Darker Wood Planks                |